<a href="https://colab.research.google.com/github/tonHS/Canadian_Real_Estate_Stats/blob/main/Copy_of_Canadian_Housing_Stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [141]:
# ============================================================================
# Install Dependencies and Packages
# ============================================================================
!pip install stats-can openpyxl

import pandas as pd
import requests
import zipfile
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import stats_can

In [142]:
# ============================================================================
# SETUP: Create directory
# ============================================================================
print(">>> ENTERING SETUP")

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
outputs_dir = Path('outputs')
outputs_dir.mkdir(exist_ok=True)

>>> ENTERING SETUP


In [143]:
# ============================================================================
# FETCH: CMA Rental Stats Data
# ============================================================================
print(">>> ENTERING: CMA Rental Data Fetch")
print("=" * 80)
print("FETCHING CMA RENTAL DATA FROM STATISTICS CANADA")
print("=" * 80)

TABLE_ID = "46100092"
download_url = f"https://www150.statcan.gc.ca/n1/tbl/csv/{TABLE_ID}-eng.zip"

print(f"\n📥 Downloading data from Statistics Canada (Table {TABLE_ID})...")
response = requests.get(download_url, timeout=30)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as zip_file:
    csv_files = [f for f in zip_file.namelist() if f.endswith('.csv')]
    csv_filename = csv_files[0]
    with zip_file.open(csv_filename) as csv_file:
        df_rents = pd.read_csv(csv_file)

print(f"✓ Data loaded: {len(df_rents):,} rows, {len(df_rents.columns)} columns")

# Save raw data
raw_data_path = data_dir / 'cma_rents_raw.csv'
df_rents.to_csv(raw_data_path, index=False)
print(f"✓ Raw data saved to: {raw_data_path}")
print(">>> CMA Employment Data Fetch COMPLETE")

df_rents.head()

>>> ENTERING: CMA Rental Data Fetch
FETCHING CMA RENTAL DATA FROM STATISTICS CANADA

📥 Downloading data from Statistics Canada (Table 46100092)...
✓ Data loaded: 8,627 rows, 16 columns
✓ Raw data saved to: data/cma_rents_raw.csv
>>> CMA Employment Data Fetch COMPLETE


,REF_DATE,GEO,DGUID,Rental unit type,Estimates,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2019-01,All census metropolitan areas,NaN,House - 3 or more bedrooms,Average asking rent,Dollars,81,units,0,v1869780581,43.7.1,2420.00,NaN,NaN,NaN,0
1,2019-01,All census metropolitan areas,NaN,Apartment - No bedroom,Average asking rent,Dollars,81,units,0,v1869780582,43.1.1,1090.00,NaN,NaN,NaN,0
2,2019-01,All census metropolitan areas,NaN,Apartment - 1 bedroom,Average asking rent,Dollars,81,units,0,v1869780583,43.2.1,1300.00,NaN,NaN,NaN,0
3,2019-01,All census metropolitan areas,NaN,Apartment - 2 bedrooms,Average asking rent,Dollars,81,units,0,v1869780584,43.3.1,1520.00,NaN,NaN,NaN,0
4,2019-01,All census metropolitan areas,NaN,Apartment - 3 or more bedrooms,Average asking rent,Dollars,81,units,0,v1869780585,43.4.1,1850.00,NaN,NaN,NaN,0


In [144]:
# ============================================================================
# PROCESS: CMA Rental Data
# ============================================================================

# Filter for the rent for studio, 1B and 2B apartments nationally and in City Stats Dashboard's 13 CMAs

dashboard_cmas = [
    'All census metropolitan areas',
    'Toronto, Census metropolitan area (CMA)',
    'Montréal, Census metropolitan area (CMA)',
    'Vancouver, Census metropolitan area (CMA)',
    'Calgary, Census metropolitan area (CMA)',
    'Edmonton, Census metropolitan area (CMA)',
    'Ottawa - Gatineau (Ontario part), Census metropolitan area (CMA)',
    'Winnipeg, Census metropolitan area (CMA)',
    'Québec, Census metropolitan area (CMA)',
    'Hamilton, Census metropolitan area (CMA)',
    'Halifax, Census metropolitan area (CMA)',
    'Saskatoon, Census metropolitan area (CMA)',
    'Fredericton, Census metropolitan area (CMA)',
    "St. John's, Census metropolitan area (CMA)"
]


# Filter df_jobs for the dashboard CMAs, employment and unemployment rate
df_rents_filtered = df_rents[
    (df_rents['GEO'].isin(dashboard_cmas)) &
    (df_rents['Rental unit type'].isin(['Apartment - No bedroom', 'Apartment - 1 bedroom', 'Apartment - 2 bedrooms', 'Room'])) &
    (df_rents['Estimates'] == 'Average asking rent') &
    (df_rents['REF_DATE'] == df_rents['REF_DATE'].max()) # This is quarterly data so check syntax if an error
].copy()

# Select relevant columns
df_rents_filtered = df_rents_filtered[[
    'REF_DATE',
    'GEO',
    'Rental unit type',
    'Estimates',
    'VALUE'
]].copy()

print(f"Filtered housing data for {len(dashboard_cmas)} dashboard CMAs for the most recent month.")
print(f"Data loaded: {len(df_rents_filtered):,} rows, {len(df_rents_filtered.columns)} columns.")

# Table 1: CMA rents for studio, 1B and 2B, 2026. Rows: GEO; Columns: Value for Each Rental Unit Type; Title(all): Estimates, Avg Asking Rent, 2024

df_rents_filtered.head(51)

Filtered housing data for 14 dashboard CMAs for the most recent month.
Data loaded: 55 rows, 5 columns.


,REF_DATE,GEO,Rental unit type,Estimates,VALUE
8329,2026-01,All census metropolitan areas,Apartment - No bedroom,Average asking rent,1420.00
8330,2026-01,All census metropolitan areas,Apartment - 1 bedroom,Average asking rent,1740.00
8331,2026-01,All census metropolitan areas,Apartment - 2 bedrooms,Average asking rent,2150.00
8335,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - No bedroom,Average asking rent,NaN
8336,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - 1 bedroom,Average asking rent,1250.00
8337,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - 2 bedrooms,Average asking rent,1570.00
8339,2026-01,"St. John's, Census metropolitan area (CMA)",Room,Average asking rent,650.00
8342,2026-01,"Halifax, Census metropolitan area (CMA)",Apartment - No bedroom,Average asking rent,1540.00
8343,2026-01,"Halifax, Census metropolitan area (CMA)",Apartment - 1 bedroom,Average asking rent,1830.00
8344,2026-01,"Halifax, Census metropolitan area (CMA)",Apartment - 2 bedrooms,Average asking rent,2350.00


In [145]:
####################################################################################################################################
# Fetch: Regional earnings current $ data
####################################################################################################################################

print(">>> Fetching latest headline weekly earnings by region (StatCan WDS)")

TABLE_ID = 14100203
WDS = "https://www150.statcan.gc.ca/t1/wds/rest"

# Map geography member IDs → names
meta = requests.post(f"{WDS}/getCubeMetadata", json=[{"productId": TABLE_ID}], timeout=60).json()[0]["object"]
geo = {m["memberId"]: m["memberNameEn"] for m in meta["dimension"][0]["member"]}

# All 14 geos · All employees · Including overtime · Industrial aggregate · latest month only
coords = [{"productId": TABLE_ID, "coordinate": f"{g}.1.1.2.0.0.0.0.0.0", "latestN": 1} for g in geo]
data = requests.post(f"{WDS}/getDataFromCubePidCoordAndLatestNPeriods", json=coords, timeout=60).json()

rows = []
for item in data:
    if item.get("status") != "SUCCESS":
        continue
    o = item["object"]; p = o["vectorDataPoint"][0]
    rows.append({"GEO": geo[int(o["coordinate"].split(".")[0])],
                 "REF_DATE": p["refPer"],
                 "WEEKLY_EARNINGS": float(p["value"])})

df_earnings = (pd.DataFrame(rows)
               .sort_values("WEEKLY_EARNINGS", ascending=False)
               .reset_index(drop=True))

raw_data_path = data_dir / 'current$_earnings_raw.csv'
df_earnings.to_csv(raw_data_path, index=False)
print(f"✓ {len(df_earnings)} regions · {df_earnings['REF_DATE'].iloc[0]} · saved to {raw_data_path}")

df_earnings.head(14)


>>> Fetching latest headline weekly earnings by region (StatCan WDS)
✓ 14 regions · 2026-03-01 · saved to data/current$_earnings_raw.csv


,GEO,REF_DATE,WEEKLY_EARNINGS
0,Nunavut,2026-03-01,1834.29
1,Northwest Territories,2026-03-01,1749.94
2,Yukon,2026-03-01,1525.38
3,Alberta,2026-03-01,1387.32
4,Ontario,2026-03-01,1379.55
5,British Columbia,2026-03-01,1357.38
6,Canada,2026-03-01,1338.14
7,Saskatchewan,2026-03-01,1296.68
8,Newfoundland and Labrador,2026-03-01,1291.80
9,Quebec,2026-03-01,1274.96


In [146]:
####################################################################################################################################
# Process: Regional wage data
####################################################################################################################################

# Create a column for monthly earnings

df_earnings["MONTHLY_EARNINGS"] = (df_earnings["WEEKLY_EARNINGS"] * 4.3).round(2)
df_earnings["APPROX_MONTHLY_TAX"] = (df_earnings["MONTHLY_EARNINGS"] * 0.15).round(2)
df_earnings["APPROX_MONTHLY_AFTERTAX_EARNINGS"] = (df_earnings["MONTHLY_EARNINGS"] - df_earnings["APPROX_MONTHLY_TAX"]).round(2)

# Table Regional Weekly and Monthly Earnings Data in Current Dollars, assuming average monthly taxes of 15% across all brackets and income levels
df_earnings.head(15)


,GEO,REF_DATE,WEEKLY_EARNINGS,MONTHLY_EARNINGS,APPROX_MONTHLY_TAX,APPROX_MONTHLY_AFTERTAX_EARNINGS
0,Nunavut,2026-03-01,1834.29,7887.45,1183.12,6704.33
1,Northwest Territories,2026-03-01,1749.94,7524.74,1128.71,6396.03
2,Yukon,2026-03-01,1525.38,6559.13,983.87,5575.26
3,Alberta,2026-03-01,1387.32,5965.48,894.82,5070.66
4,Ontario,2026-03-01,1379.55,5932.06,889.81,5042.25
5,British Columbia,2026-03-01,1357.38,5836.73,875.51,4961.22
6,Canada,2026-03-01,1338.14,5754.00,863.10,4890.90
7,Saskatchewan,2026-03-01,1296.68,5575.72,836.36,4739.36
8,Newfoundland and Labrador,2026-03-01,1291.80,5554.74,833.21,4721.53
9,Quebec,2026-03-01,1274.96,5482.33,822.35,4659.98


In [147]:
###################################################################################################################################
# Analysis - Avg CMA rents for studio, 1B and 2B compared to regional earning (national and provincial)
####################################################################################################################################

# Table: Insight into proportion of regional monthly earnings would be taken up by rent

# Join df_earnings to df_rents_filtered in such a way that the correct provincial after-tax earnings are added to the corresponding CMA rows. This will then allow calculation of CMA rent's % of average provincial earnings.

# Define the cma_to_prov_map dictionary
cma_to_prov_map = {
    'All census metropolitan areas': 'Canada',
    'Toronto, Census metropolitan area (CMA)': 'Ontario',
    'Montréal, Census metropolitan area (CMA)': 'Quebec',
    'Vancouver, Census metropolitan area (CMA)': 'British Columbia',
    'Calgary, Census metropolitan area (CMA)': 'Alberta',
    'Edmonton, Census metropolitan area (CMA)': 'Alberta',
    'Ottawa-Gatineau, Ontario/Quebec, Census metropolitan area (CMA)': 'Ontario', # Assuming primary province for simplicity
    'Winnipeg, Census metropolitan area (CMA)': 'Manitoba',
    'Québec, Census metropolitan area (CMA)': 'Quebec',
    'Hamilton, Census metropolitan area (CMA)': 'Ontario',
    'Halifax, Census metropolitan area (CMA)': 'Nova Scotia',
    'Saskatoon, Census metropolitan area (CMA)': 'Saskatchewan',
    'Fredericton, Census metropolitan area (CMA)': 'New Brunswick',
    "St. John's, Census metropolitan area (CMA)": 'Newfoundland and Labrador'
}

# Step 1: Create a 'Province' column in df_rents_filtered by mapping CMA names to their provinces.
df_rents_filtered['Province'] = df_rents_filtered['GEO'].map(cma_to_prov_map)

df_rents_filtered.head()



,REF_DATE,GEO,Rental unit type,Estimates,VALUE,Province
8329,2026-01,All census metropolitan areas,Apartment - No bedroom,Average asking rent,1420.00,Canada
8330,2026-01,All census metropolitan areas,Apartment - 1 bedroom,Average asking rent,1740.00,Canada
8331,2026-01,All census metropolitan areas,Apartment - 2 bedrooms,Average asking rent,2150.00,Canada
8335,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - No bedroom,Average asking rent,NaN,Newfoundland and Labrador
8336,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - 1 bedroom,Average asking rent,1250.00,Newfoundland and Labrador


In [148]:
# Analysis continued

# Step 2: Merge df_rents_filtered with df_earnings to bring in the after-tax monthly earnings.
# We need to merge on the 'Province' column from df_rents_filtered and the 'GEO' column from df_earnings.
# We will use a left merge to keep all rows from df_rents_filtered and add matching earnings.
df_rents_filtered = pd.merge(
    df_rents_filtered,
    df_earnings[['GEO', 'APPROX_MONTHLY_AFTERTAX_EARNINGS']],
    left_on='Province',
    right_on='GEO',
    how='left',
    suffixes=('_rent', '_earnings')
)

# Step 3: Clean up and rename the new earnings column.
# The 'GEO_earnings' column is redundant after the merge, so we drop it
# df_rents_filtered = df_rents_filtered.drop(columns=['GEO_earnings'])
df_rents_filtered = df_rents_filtered.rename(
    columns={'APPROX_MONTHLY_AFTERTAX_EARNINGS': 'Approx Provincial After-tax Monthly Earnings'}
)

# Step 4: Calculate the proportion of regional monthly earnings taken up by rent.
# The 'VALUE' column contains the average asking rent.
df_rents_filtered['Rent as % of Provincial After-tax Monthly Earnings'] = (
    (df_rents_filtered['VALUE'] / df_rents_filtered['Approx Provincial After-tax Monthly Earnings']) * 100
).round(2)

# Display the head of the modified df_rents_filtered to verify
print("Modified df_rents_filtered with provincial earnings and rent as percentage:")
# print(f"Data loaded: {len(df_rents_filtered):,} rows, {len(df_rents_filtered.columns)} columns.")
df_rents_filtered.head(30)


Modified df_rents_filtered with provincial earnings and rent as percentage:


,REF_DATE,GEO_rent,Rental unit type,Estimates,VALUE,Province,GEO_earnings,Approx Provincial After-tax Monthly Earnings,Rent as % of Provincial After-tax Monthly Earnings
0,2026-01,All census metropolitan areas,Apartment - No bedroom,Average asking rent,1420.00,Canada,Canada,4890.90,29.03
1,2026-01,All census metropolitan areas,Apartment - 1 bedroom,Average asking rent,1740.00,Canada,Canada,4890.90,35.58
2,2026-01,All census metropolitan areas,Apartment - 2 bedrooms,Average asking rent,2150.00,Canada,Canada,4890.90,43.96
3,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - No bedroom,Average asking rent,NaN,Newfoundland and Labrador,Newfoundland and Labrador,4721.53,NaN
4,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - 1 bedroom,Average asking rent,1250.00,Newfoundland and Labrador,Newfoundland and Labrador,4721.53,26.47
5,2026-01,"St. John's, Census metropolitan area (CMA)",Apartment - 2 bedrooms,Average asking rent,1570.00,Newfoundland and Labrador,Newfoundland and Labrador,4721.53,33.25
6,2026-01,"St. John's, Census metropolitan area (CMA)",Room,Average asking rent,650.00,Newfoundland and Labrador,Newfoundland and Labrador,4721.53,13.77
7,2026-01,"Halifax, Census metropolitan area (CMA)",Apartment - No bedroom,Average asking rent,1540.00,Nova Scotia,Nova Scotia,4404.72,34.96
8,2026-01,"Halifax, Census metropolitan area (CMA)",Apartment - 1 bedroom,Average asking rent,1830.00,Nova Scotia,Nova Scotia,4404.72,41.55
9,2026-01,"Halifax, Census metropolitan area (CMA)",Apartment - 2 bedrooms,Average asking rent,2350.00,Nova Scotia,Nova Scotia,4404.72,53.35


In [149]:
####################################################################################################################################
# Filtering to create new object (data frame? things?) without unnecessary columns
####################################################################################################################################

earnings_rent_viz = df_rents_filtered[[
    'REF_DATE',
    'GEO_rent',
    'Rental unit type',
    'VALUE',
    'Estimates',
    'Approx Provincial After-tax Monthly Earnings',
    'Rent as % of Provincial After-tax Monthly Earnings'
]].copy()

# Naming columns more clearly

earnings_rent_viz = earnings_rent_viz.rename(columns={"GEO_rent": "Census Metropolitan Area", "VALUE": "Rent Value"})

# earnings_rent_viz.head()

earnings_rent_viz_mini = earnings_rent_viz[[
    'Census Metropolitan Area',
    'Rental unit type',
    'Rent Value',
    'Rent as % of Provincial After-tax Monthly Earnings'
]].copy()

# Explanatory note: the after-tax monthly earnings are an estimate that assumes taxes are 15% on monthly average provincial earnings.
# Analysis would be more precise and accurate with CMA-specific earnings data and non-proxy after-tax data.
# Verification through a quick spotcheck after everything is done would be a quick precautionary step

earnings_rent_viz_mini.head(70)

,Census Metropolitan Area,Rental unit type,Rent Value,Rent as % of Provincial After-tax Monthly Earnings
0,All census metropolitan areas,Apartment - No bedroom,1420.00,29.03
1,All census metropolitan areas,Apartment - 1 bedroom,1740.00,35.58
2,All census metropolitan areas,Apartment - 2 bedrooms,2150.00,43.96
3,"St. John's, Census metropolitan area (CMA)",Apartment - No bedroom,NaN,NaN
4,"St. John's, Census metropolitan area (CMA)",Apartment - 1 bedroom,1250.00,26.47
5,"St. John's, Census metropolitan area (CMA)",Apartment - 2 bedrooms,1570.00,33.25
6,"St. John's, Census metropolitan area (CMA)",Room,650.00,13.77
7,"Halifax, Census metropolitan area (CMA)",Apartment - No bedroom,1540.00,34.96
8,"Halifax, Census metropolitan area (CMA)",Apartment - 1 bedroom,1830.00,41.55
9,"Halifax, Census metropolitan area (CMA)",Apartment - 2 bedrooms,2350.00,53.35
